#f

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
PROJECT_PATH = "/content/drive/MyDrive/MedGraph-AI-20260813T103118Z-1-001.zip"

DATA_PATH = f"{PROJECT_PATH}/data/raw"

print(DATA_PATH)

/content/drive/MyDrive/MedGraph-AI-20260813T103118Z-1-001.zip/data/raw


In [5]:
from google.colab import files
import os

uploaded = files.upload()

Saving MedGraph-AI-20260813T103118Z-1-001.zip to MedGraph-AI-20260813T103118Z-1-001.zip


In [6]:
import zipfile
import os

zip_path = "/content/data.zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Done!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/data.zip'

In [ ]:
from google.colab import drive
import shutil
import os

# Connect Google Drive
drive.mount('/content/drive')

# Source
source = "/content/data"

# Destination
destination = "/content/drive/MyDrive/MedGraph-AI/data"

# Create destination parent folder
os.makedirs(os.path.dirname(destination), exist_ok=True)

# If destination already exists, remove it first
if os.path.exists(destination):
    shutil.rmtree(destination)

# Copy the whole data folder
shutil.copytree(source, destination)

print("✅ Data copied successfully!")
print(f"📁 Location: {destination}")

In [ ]:
import os

DATA_PATH = "/content/drive/MyDrive/MedGraph-AI/data"

for root, dirs, files in os.walk(DATA_PATH):
    level = root.replace(DATA_PATH, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")
        data/
    USPSTF/
        06_healthy_diet_physical_activity_no_risk_2022.pdf
        01_statins_2022.pdf
        02_hypertension_2021.pdf
        03_prediabetes_diabetes_2021.pdf
        04_aspirin_cvd_2022.pdf
        New DOCX Document.docx
        05_healthy_diet_physical_activity_risk_2020.pdf
    WHO/
        HEARTS_Risk-based_CVD_management.pdf
        HEARTS_Healthy-lifestyle_counselling.pdf
        HEARTS_Evidence-based_treatment_protocols.pdf
    ESC/
        ESC_Guidelines_on_Cardiovascular_Disease_Prevention_in_Clinical_Practice.pdf
        Focused_Update_of_ESC_EAS_Dyslipidaemias.pdf

In [ ]:
# في Colab، استخدم os.environ أو getpass بدل ما تكتب الباسورد في الكود
from getpass import getpass
import os

os.environ["NEO4J_URI"] = "neo4j+s://68888074.databases.neo4j.io"
os.environ["NEO4J_USERNAME"] = "68888074"
os.environ["NEO4J_PASSWORD"] = getpass("Enter Neo4j password: ")

Enter Neo4j password: ··········


In [ ]:
import os

DATA_PATH = "/content/drive/MyDrive/MedGraph-AI/data"

# Only these source folders are included in the corpus
INCLUDED_SOURCES = ["USPSTF", "WHO"]

# Explicit file exclusions (non-guideline content)
EXCLUDED_FILES = {"New DOCX Document.docx"}

# File extensions we can actually parse
ALLOWED_EXTENSIONS = {".pdf"}

def collect_pdf_paths(data_path, included_sources, excluded_files):
    collected = []
    for source in included_sources:
        source_dir = os.path.join(data_path, source)
        if not os.path.isdir(source_dir):
            print(f"⚠️  Warning: {source_dir} not found, skipping.")
            continue
        for fname in sorted(os.listdir(source_dir)):
            fpath = os.path.join(source_dir, fname)
            ext = os.path.splitext(fname)[1].lower()
            if fname in excluded_files:
                continue
            if ext not in ALLOWED_EXTENSIONS:
                continue
            collected.append({"source": source, "document": fname, "path": fpath})
    return collected

pdf_files = collect_pdf_paths(DATA_PATH, INCLUDED_SOURCES, EXCLUDED_FILES)

print(f"✅ Found {len(pdf_files)} PDFs to ingest:\n")
for f in pdf_files:
    print(f"  [{f['source']}] {f['document']}")

✅ Found 9 PDFs to ingest:

  [USPSTF] 01_statins_2022.pdf
  [USPSTF] 02_hypertension_2021.pdf
  [USPSTF] 03_prediabetes_diabetes_2021.pdf
  [USPSTF] 04_aspirin_cvd_2022.pdf
  [USPSTF] 05_healthy_diet_physical_activity_risk_2020.pdf
  [USPSTF] 06_healthy_diet_physical_activity_no_risk_2022.pdf
  [WHO] HEARTS_Evidence-based_treatment_protocols.pdf
  [WHO] HEARTS_Healthy-lifestyle_counselling.pdf
  [WHO] HEARTS_Risk-based_CVD_management.pdf


#statrt

In [ ]:
#@title 0) Configuration
from pathlib import Path

CONFIG = {
    "project_dir": "/content/drive/MyDrive/MedGraph-AI",
    "included_sources": ["USPSTF", "WHO"],
    "min_chars_per_page": 80,
    "chunk_max_chars": 1800,
    "chunk_overlap": 250,
    "embedding_model": "intfloat/multilingual-e5-large-instruct",
    "chroma_collection": "guidelines",
    "default_top_k": 10,
    "min_supporting_hits": 2,
    # Knowledge-graph extraction
    "kg_source_sections": ["Recommendation", "Population", "Evidence Assessment", "Practice Considerations"],
    "kg_max_chunks": None,  # set an int while testing/iterating to limit LLM calls
    "groq_model": "llama-3.1-8b-instant",
    # Hybrid retrieval
    "graph_retrieval_limit": 15,
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "rerank_top_n": 10,
}

PROJECT_DIR = Path(CONFIG["project_dir"])
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"


In [ ]:
#@title 1) Install dependencies
!pip -q install pymupdf chromadb sentence-transformers neo4j supabase python-dotenv pandas numpy tqdm scikit-learn groq


In [ ]:
#@title 2) Mount Google Drive and define paths
from google.colab import drive
import os, re, json, hashlib
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

drive.mount('/content/drive')

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
USPSTF_DIR = DATA_DIR / 'USPSTF'
WHO_DIR = DATA_DIR / 'WHO'

print('Project:', PROJECT_DIR)
print('USPSTF PDFs:', len(list(USPSTF_DIR.rglob('*.pdf'))) if USPSTF_DIR.exists() else 0)
print('WHO PDFs:', len(list(WHO_DIR.rglob('*.pdf'))) if WHO_DIR.exists() else 0)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/MedGraph-AI
USPSTF PDFs: 6
WHO PDFs: 3


In [ ]:
#@title 2A) Audit accepted and excluded files
included = set(CONFIG['included_sources'])
accepted, excluded = [], []

for path in DATA_DIR.rglob('*'):
    if not path.is_file():
        continue
    relative = path.relative_to(DATA_DIR)
    parts = relative.parts
    if len(parts) >= 2 and parts[0] in included and path.suffix.lower() == '.pdf':
        accepted.append({'source': parts[0], 'file': str(relative), 'size_mb': round(path.stat().st_size / 1e6, 2)})
    else:
        excluded.append({'file': str(relative), 'reason': f'Only {sorted(included)} PDF files are in scope'})

file_audit = pd.DataFrame(accepted, columns=['source', 'file', 'size_mb']).sort_values(['source', 'file'])
excluded_audit = pd.DataFrame(excluded, columns=['file', 'reason']).sort_values('file')

print('Accepted files:', len(file_audit))
print(file_audit.to_string(index=False))
print('\nExcluded files:', len(excluded_audit))
print(excluded_audit.to_string(index=False))

file_audit.to_csv(ARTIFACT_DIR / 'accepted_files.csv', index=False)
excluded_audit.to_csv(ARTIFACT_DIR / 'excluded_files.csv', index=False)


Accepted files: 9
source                                                      file  size_mb
USPSTF                                USPSTF/01_statins_2022.pdf     0.51
USPSTF                           USPSTF/02_hypertension_2021.pdf     0.51
USPSTF                   USPSTF/03_prediabetes_diabetes_2021.pdf     0.17
USPSTF                            USPSTF/04_aspirin_cvd_2022.pdf     0.58
USPSTF    USPSTF/05_healthy_diet_physical_activity_risk_2020.pdf     0.48
USPSTF USPSTF/06_healthy_diet_physical_activity_no_risk_2022.pdf     0.50
   WHO         WHO/HEARTS_Evidence-based_treatment_protocols.pdf     1.74
   WHO              WHO/HEARTS_Healthy-lifestyle_counselling.pdf     2.27
   WHO                  WHO/HEARTS_Risk-based_CVD_management.pdf     2.91

Excluded files: 3
                                                                            file                                        reason
ESC/ESC_Guidelines_on_Cardiovascular_Disease_Prevention_in_Clinical_Practice.pdf Only ['USPSTF',

In [ ]:
#@title 3) Load credentials from Colab Secrets
# Left sidebar -> key icon -> Secrets -> add each variable, then re-run this cell.
# Required: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD
# Optional:  NEO4J_DATABASE, SUPABASE_URL, SUPABASE_PUBLISHABLE_KEY, GROQ_API_KEY

try:
    from google.colab import userdata
    def secret(name, required=False):
        value = userdata.get(name)
        if required and not value:
            raise RuntimeError(f'Missing Colab Secret: {name}')
        return value
except Exception:
    def secret(name, required=False):
        value = os.getenv(name)
        if required and not value:
            raise RuntimeError(f'Missing environment variable: {name}')
        return value

NEO4J_URI = secret('NEO4J_URI', required=True)
NEO4J_USERNAME = secret('NEO4J_USERNAME', required=True)
NEO4J_PASSWORD = secret('NEO4J_PASSWORD', required=True)
NEO4J_DATABASE = secret('NEO4J_DATABASE') or 'neo4j'

SUPABASE_URL = secret('SUPABASE_URL')
SUPABASE_KEY = secret('SUPABASE_PUBLISHABLE_KEY')
GROQ_API_KEY = secret('GROQ_API_KEY')  # optional, enables stages 9 and 16

print('Neo4j URI:', NEO4J_URI)
print('Neo4j database:', NEO4J_DATABASE)
print('Supabase configured:', bool(SUPABASE_URL and SUPABASE_KEY))
print('Groq (KG extraction + grounded answers):', 'enabled' if GROQ_API_KEY else 'disabled (optional; rule-based fallback will be used)')


Neo4j URI: neo4j+s://68888074.databases.neo4j.io
Neo4j database: 68888074
Supabase configured: True
Groq (KG extraction + grounded answers): enabled


In [ ]:
#@title 4) Verify Neo4j and optional Supabase connections
from neo4j import GraphDatabase

neo4j_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    connection_timeout=30,
)
neo4j_driver.verify_connectivity()
with neo4j_driver.session(database=NEO4J_DATABASE) as session:
    assert session.run('RETURN 1 AS ok').single()['ok'] == 1
print('Neo4j connection: OK')

supabase = None
if SUPABASE_URL and SUPABASE_KEY:
    from supabase import create_client
    supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
    print('Supabase client: initialized')
else:
    print('Supabase: skipped (optional)')


Neo4j connection: OK
Supabase client: initialized


In [ ]:
WHO_YEARS = {
    "HEARTS_Evidence-based_treatment_protocols.pdf": 2018,
    "HEARTS_Healthy-lifestyle_counselling.pdf": 2018,
    "HEARTS_Risk-based_CVD_management.pdf": 2020,
}

In [ ]:
#@title 5) Extract pages from USPSTF and WHO PDFs
import pymupdf
import re
import pandas as pd

def infer_year(filename):
    match = re.search(r'(?<!\d)(19|20)\d{2}(?!\d)', filename)
    return int(match.group(0)) if match else None


def ingest_pdf(pdf_path, source):
    rows = []
    year = infer_year(pdf_path.name)

    if year is None and source == "WHO":
        year = WHO_YEARS.get(pdf_path.name)
    with pymupdf.open(pdf_path) as doc:

        for page_number, page in enumerate(doc, start=1):

            text = page.get_text("text") or ""
            images = page.get_images(full=True)

            # Keep the page if it has text OR images
            if text.strip() or images:

                rows.append({
                    "text": text.strip(),
                    "source": source,
                    "document": pdf_path.name,
                    "path": str(pdf_path),
                    "page": page_number,
                    # "year": infer_year(pdf_path.name),
                    "year": year,
                    "has_images": len(images) > 0,
                    "image_count": len(images),
                })

    return rows


records = []

for source in CONFIG["included_sources"]:

    root = DATA_DIR / source

    if root.exists():

        for pdf in sorted(root.rglob("*.pdf")):
            records.extend(
                ingest_pdf(pdf, source)
            )

    else:
        print(f"⚠️ {root} not found, skipping.")


documents_df = pd.DataFrame(records)

if documents_df.empty:
    raise FileNotFoundError(
        "No PDF pages found. Check the Google Drive folders."
    )


missing_years = sorted(
    documents_df.loc[
        documents_df["year"].isna(),
        "document"
    ].unique()
)

if missing_years:
    print(
        "⚠️ Could not infer a year from the filename for:",
        missing_years
    )


documents_df.to_json(
    ARTIFACT_DIR / "documents.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)


print("Shape:", documents_df.shape)

print(
    "Pages with images:",
    documents_df["has_images"].sum()
)

display(documents_df.head(8))

Shape: (196, 8)
Pages with images: 19


,text,source,document,path,page,year,has_images,image_count
0,Statin Use for the Primary Prevention of Cardi...,USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,1,2022,False,0
1,USPSTF Assessment of Magnitude of Net Benefit\...,USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,2,2022,False,0
2,"dyslipidemia, diabetes, or hypertension, the U...",USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,3,2022,False,0
3,compared with placebo.13 Almost all trials did...,USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,4,2022,False,0
4,Other Related USPSTF Recommendations\nTheUSPST...,USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,5,2022,False,0
5,There were also limited data directly comparin...,USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,6,2022,False,0
6,Equations and inform decision-making for perso...,USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,7,2022,False,0
7,"18. Wall HK, Ritchey MD, Gillespie C, Omura JD...",USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,8,2022,False,0


In [ ]:
#@title 6) Inspect pages containing images

image_pages = documents_df[
    documents_df["has_images"] == True
][[
    "source",
    "document",
    "page",
    "year",
    "image_count",
    "text"
]].copy()

print(f"Total pages with images: {len(image_pages)}")

display(
    image_pages[
        [
            "source",
            "document",
            "page",
            "year",
            "image_count"
        ]
    ].sort_values(
        ["source", "document", "page"]
    ).reset_index(drop=True)
)

Total pages with images: 19


,source,document,page,year,image_count
0,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,1,2018,7
1,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,3,2018,7
2,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,17,2018,1
3,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,19,2018,1
4,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,21,2018,1
5,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,23,2018,1
6,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,25,2018,1
7,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,27,2018,1
8,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,29,2018,1
9,WHO,HEARTS_Healthy-lifestyle_counselling.pdf,1,2018,7


In [ ]:
#@title 6A) Preview WHO image pages

from IPython.display import display, Image as IPImage
import pymupdf
from pathlib import Path

PREVIEW_DIR = ARTIFACT_DIR / "image_previews"
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
#@title 6) Inspect pages containing images

image_pages = documents_df[
    documents_df["has_images"] == True
][[
    "source",
    "document",
    "path",
    "page",
    "year",
    "image_count",
    "text"
]].copy()

print(f"Total pages with images: {len(image_pages)}")

display(
    image_pages[
        [
            "source",
            "document",
            "page",
            "year",
            "image_count"
        ]
    ]
    .sort_values(
        ["source", "document", "page"]
    )
    .reset_index(drop=True)
)

Total pages with images: 19


,source,document,page,year,image_count
0,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,1,2018,7
1,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,3,2018,7
2,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,17,2018,1
3,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,19,2018,1
4,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,21,2018,1
5,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,23,2018,1
6,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,25,2018,1
7,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,27,2018,1
8,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,29,2018,1
9,WHO,HEARTS_Healthy-lifestyle_counselling.pdf,1,2018,7


# t

In [ ]:
#@title 7A) Restore project paths

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/MedGraph-AI")
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

IMAGE_DIR = ARTIFACT_DIR / "images"

IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("✅ Paths restored")
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("IMAGE_DIR:", IMAGE_DIR)
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/MedGraph-AI")
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
IMAGE_DIR = ARTIFACT_DIR / "images"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Paths restored")

✅ Paths restored
PROJECT_DIR: /content/drive/MyDrive/MedGraph-AI
DATA_DIR: /content/drive/MyDrive/MedGraph-AI/data
ARTIFACT_DIR: /content/drive/MyDrive/MedGraph-AI/artifacts
IMAGE_DIR: /content/drive/MyDrive/MedGraph-AI/artifacts/images
✅ Paths restored


In [ ]:
import pymupdf
import re
import pandas as pd

CONFIG = {
    "included_sources": ["USPSTF", "WHO"]
}

WHO_YEARS = {
    "HEARTS_Evidence-based_treatment_protocols.pdf": 2018,
    "HEARTS_Healthy-lifestyle_counselling.pdf": 2018,
    "HEARTS_Risk-based_CVD_management.pdf": 2020,
}

def infer_year(filename):
    match = re.search(r'(?<!\d)(19|20)\d{2}(?!\d)', filename)
    return int(match.group(0)) if match else None

def ingest_pdf(pdf_path, source):
    rows = []

    year = infer_year(pdf_path.name)

    if year is None and source == "WHO":
        year = WHO_YEARS.get(pdf_path.name)

    with pymupdf.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):

            text = page.get_text("text") or ""
            images = page.get_images(full=True)

            if text.strip() or images:
                rows.append({
                    "text": text.strip(),
                    "source": source,
                    "document": pdf_path.name,
                    "path": str(pdf_path),
                    "page": page_number,
                    "year": year,
                    "has_images": len(images) > 0,
                    "image_count": len(images),
                })

    return rows

records = []

for source in CONFIG["included_sources"]:
    root = DATA_DIR / source

    if root.exists():
        for pdf in sorted(root.rglob("*.pdf")):
            records.extend(ingest_pdf(pdf, source))
    else:
        print(f"⚠️ {root} not found")

documents_df = pd.DataFrame(records)

print("Shape:", documents_df.shape)
print("Pages with images:", documents_df["has_images"].sum())

display(documents_df.head(2))

Shape: (196, 8)
Pages with images: 19


,text,source,document,path,page,year,has_images,image_count
0,Statin Use for the Primary Prevention of Cardi...,USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,1,2022,False,0
1,USPSTF Assessment of Magnitude of Net Benefit\...,USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,2,2022,False,0


In [ ]:
#@title 7A) Extract all embedded images from WHO/USPSTF PDFs

from pathlib import Path
import pymupdf
import pandas as pd

IMAGE_DIR = ARTIFACT_DIR / "images"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

image_records = []

for _, row in documents_df[
    documents_df["has_images"] == True
].iterrows():

    pdf_path = Path(row["path"])
    page_number = int(row["page"])

    with pymupdf.open(pdf_path) as doc:

        page = doc[page_number - 1]

        page_images = page.get_images(full=True)

        for image_index, image_info in enumerate(page_images):

            xref = image_info[0]

            try:
                base = doc.extract_image(xref)

                ext = base["ext"]

                image_name = (
                    f"{pdf_path.stem}"
                    f"_page_{page_number:03d}"
                    f"_img_{image_index:02d}.{ext}"
                )

                image_path = IMAGE_DIR / image_name

                image_path.write_bytes(
                    base["image"]
                )

                image_records.append({

                    "image_id": (
                        f"{pdf_path.stem}"
                        f"_p{page_number}"
                        f"_i{image_index}"
                    ),

                    "source": row["source"],

                    "document": row["document"],

                    "path": str(pdf_path),

                    "page": page_number,

                    "image_index": image_index,

                    "image_path": str(image_path),

                    "extension": ext,

                })

            except Exception as e:

                print(
                    f"⚠️ Failed: "
                    f"{row['document']} "
                    f"page {page_number} "
                    f"image {image_index}: "
                    f"{e}"
                )


images_df = pd.DataFrame(image_records)

print("=" * 60)
print("IMAGE EXTRACTION COMPLETE")
print("=" * 60)

print("Total images:", len(images_df))

display(
    images_df[
        [
            "source",
            "document",
            "page",
            "image_index",
            "extension",
            "image_path"
        ]
    ].head(20)
)

IMAGE EXTRACTION COMPLETE
Total images: 59


,source,document,page,image_index,extension,image_path
0,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,1,0,png,/content/drive/MyDrive/MedGraph-AI/artifacts/i...
1,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,1,1,png,/content/drive/MyDrive/MedGraph-AI/artifacts/i...
2,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,1,2,png,/content/drive/MyDrive/MedGraph-AI/artifacts/i...
3,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,1,3,png,/content/drive/MyDrive/MedGraph-AI/artifacts/i...
4,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,1,4,png,/content/drive/MyDrive/MedGraph-AI/artifacts/i...
5,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,1,5,png,/content/drive/MyDrive/MedGraph-AI/artifacts/i...
6,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,1,6,png,/content/drive/MyDrive/MedGraph-AI/artifacts/i...
7,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,3,0,png,/content/drive/MyDrive/MedGraph-AI/artifacts/i...
8,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,3,1,png,/content/drive/MyDrive/MedGraph-AI/artifacts/i...
9,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,3,2,png,/content/drive/MyDrive/MedGraph-AI/artifacts/i...


In [ ]:
#@title 7C) Check images_df

print(images_df.columns.tolist())
print(images_df.shape)

display(images_df.head())

['image_id', 'source', 'document', 'path', 'page', 'image_index', 'image_path', 'extension']
(59, 8)


,image_id,source,document,path,page,image_index,image_path,extension
0,HEARTS_Evidence-based_treatment_protocols_p1_i0,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,/content/drive/MyDrive/MedGraph-AI/data/WHO/HE...,1,0,/content/drive/MyDrive/MedGraph-AI/artifacts/i...,png
1,HEARTS_Evidence-based_treatment_protocols_p1_i1,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,/content/drive/MyDrive/MedGraph-AI/data/WHO/HE...,1,1,/content/drive/MyDrive/MedGraph-AI/artifacts/i...,png
2,HEARTS_Evidence-based_treatment_protocols_p1_i2,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,/content/drive/MyDrive/MedGraph-AI/data/WHO/HE...,1,2,/content/drive/MyDrive/MedGraph-AI/artifacts/i...,png
3,HEARTS_Evidence-based_treatment_protocols_p1_i3,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,/content/drive/MyDrive/MedGraph-AI/data/WHO/HE...,1,3,/content/drive/MyDrive/MedGraph-AI/artifacts/i...,png
4,HEARTS_Evidence-based_treatment_protocols_p1_i4,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,/content/drive/MyDrive/MedGraph-AI/data/WHO/HE...,1,4,/content/drive/MyDrive/MedGraph-AI/artifacts/i...,png


🔐 إعداد الكوكيز لـ Gemini


NameError: name 'chunks_df' is not defined

In [ ]:
#@title 5) Extract pages from USPSTF and WHO PDFs
import fitz

def infer_year(filename):
    match = re.search(r'(?<!\d)(19|20)\d{2}(?!\d)', filename)
    return int(match.group(0)) if match else None

def ingest_pdf(pdf_path, source):
    rows = []
    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text('text') or ''
            if text.strip():
                rows.append({
                    'text': text,
                    'source': source,
                    'document': pdf_path.name,
                    'path': str(pdf_path),
                    'page': page_number,
                    'year': infer_year(pdf_path.name),
                })
    return rows

records = []
for source in CONFIG['included_sources']:
    root = DATA_DIR / source
    if root.exists():
        for pdf in sorted(root.rglob('*.pdf')):
            records.extend(ingest_pdf(pdf, source))
    else:
        print(f'⚠️  {root} not found, skipping.')

documents_df = pd.DataFrame(records)
if documents_df.empty:
    raise FileNotFoundError('No PDF pages found. Check the Google Drive folders.')

missing_years = sorted(documents_df.loc[documents_df['year'].isna(), 'document'].unique())
if missing_years:
    print('⚠️  Could not infer a year from the filename for:', missing_years)

documents_df.to_json(ARTIFACT_DIR / 'documents.jsonl', orient='records', lines=True, force_ascii=False)
print(documents_df.shape)
documents_df.head(2)


⚠️  Could not infer a year from the filename for: ['HEARTS_Evidence-based_treatment_protocols.pdf', 'HEARTS_Healthy-lifestyle_counselling.pdf', 'HEARTS_Risk-based_CVD_management.pdf']
(195, 6)


,text,source,document,path,page,year
0,Statin Use for the Primary Prevention of Cardi...,USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,1,2022.0
1,USPSTF Assessment of Magnitude of Net Benefit\...,USPSTF,01_statins_2022.pdf,/content/drive/MyDrive/MedGraph-AI/data/USPSTF...,2,2022.0


In [ ]:
CONFIG = {
    "project_dir": "/content/drive/MyDrive/MedGraph-AI",
    "included_sources": ["USPSTF", "WHO"],
    "min_chars_per_page": 80,
    "chunk_max_chars": 1800,
    "chunk_overlap": 250,
    "embedding_model": "intfloat/multilingual-e5-large-instruct",
    "chroma_collection": "guidelines",
    "default_top_k": 10,
    "min_supporting_hits": 2,
    "kg_source_sections": ["Recommendation", "Population", "Evidence Assessment", "Practice Considerations"],
    "kg_max_chunks": None,
    "groq_model": "llama-3.1-8b-instant",
    "graph_retrieval_limit": 15,
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "rerank_top_n": 10,
}

#@title 6) Clean text and detect guideline sections
def clean_text(text):
    text = text.replace('\u00ad', '')            # soft hyphen
    text = re.sub(r'-\s*\n\s*', '', text)         # de-hyphenate across line breaks
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)  # single newlines -> space
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def detect_section(text):
    patterns = [
        ('Recommendation', r'\brecommendation(s)?\b'),
        ('Population', r'\bpopulation\b|\bpersons? aged\b'),
        ('Evidence Assessment', r'\bevidence\b|\bcertainty\b'),
        ('Practice Considerations', r'\bpractice considerations?\b'),
        ('Benefits', r'\bbenefits?\b'),
        ('Harms', r'\bharms?\b'),
        ('References', r'\breferences?\b'),
    ]
    for label, pattern in patterns:
        if re.search(pattern, text, flags=re.I):
            return label
    return 'General'

documents_df['clean_text'] = documents_df['text'].map(clean_text)
documents_df['section'] = documents_df['clean_text'].map(detect_section)
documents_df = documents_df[documents_df['clean_text'].str.len() >= CONFIG['min_chars_per_page']].reset_index(drop=True)

documents_df.to_json(ARTIFACT_DIR / 'clean_documents.jsonl', orient='records', lines=True, force_ascii=False)
print(documents_df['source'].value_counts())
print(documents_df['section'].value_counts().head())

source
WHO       145
USPSTF     46
Name: count, dtype: int64
section
General                86
Recommendation         54
Evidence Assessment    32
Population              9
References              5
Name: count, dtype: int64


In [ ]:
#@title 7) Section-aware chunking
def chunk_text(text, max_chars=CONFIG['chunk_max_chars'], overlap=CONFIG['chunk_overlap']):
    text = text.strip()
    if len(text) <= max_chars:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            boundary = max(text.rfind('. ', start, end), text.rfind('\n', start, end))
            if boundary > start + max_chars // 2:
                end = boundary + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks

chunk_rows = []
for _, row in documents_df.iterrows():
    for idx, chunk in enumerate(chunk_text(row['clean_text'])):
        chunk_id = hashlib.sha256(f"{row['source']}|{row['document']}|{row['page']}|{idx}|{chunk}".encode()).hexdigest()[:20]
        chunk_rows.append({
            'id': chunk_id,
            'text': chunk,
            'source': row['source'],
            'document': row['document'],
            'page': int(row['page']),
            'year': row['year'],
            'section': row['section'],
            'chunk_index': idx,
        })

chunks_df = pd.DataFrame(chunk_rows)
chunks_df.to_json(ARTIFACT_DIR / 'chunks.json', orient='records', force_ascii=False, indent=2)
print('Chunks:', len(chunks_df))
chunks_df.head(2)


Chunks: 510


,id,text,source,document,page,year,section,chunk_index
0,e70669dedea61c538f3b,Statin Use for the Primary Prevention of Cardi...,USPSTF,01_statins_2022.pdf,1,2022.0,Recommendation,0
1,af5972fbfc4119a96a19,ed a review of the evidence on the benefits an...,USPSTF,01_statins_2022.pdf,1,2022.0,Recommendation,1


In [ ]:
#@title 8) Embeddings and ChromaDB
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer(CONFIG['embedding_model'])

def embed_documents(texts):
    return embedder.encode(
        [f'passage: {t}' for t in texts],
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=16,
    ).tolist()

chroma_client = chromadb.PersistentClient(path=str(ARTIFACT_DIR / 'chroma'))
collection = chroma_client.get_or_create_collection(
    name=CONFIG['chroma_collection'],
    metadata={'description': 'USPSTF and WHO guideline chunks'},
)

# Rebuild is deterministic and avoids stale records when source PDFs change.
if collection.count():
    chroma_client.delete_collection(CONFIG['chroma_collection'])
    collection = chroma_client.get_or_create_collection(name=CONFIG['chroma_collection'])

embeddings = embed_documents(chunks_df['text'].tolist())
collection.add(
    ids=chunks_df['id'].tolist(),
    documents=chunks_df['text'].tolist(),
    embeddings=embeddings,
    metadatas=chunks_df[['source', 'document', 'page', 'year', 'section', 'chunk_index']].fillna('').to_dict('records'),
)
print('Chroma records:', collection.count())


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Chroma records: 510


In [ ]:
# #@title 9) Entity and relationship extraction
# ORGANIZATION_MAP = {'USPSTF': 'US Preventive Services Task Force', 'WHO': 'World Health Organization'}
# CONDITION_VOCAB = ['cardiovascular disease', 'coronary heart disease', 'stroke', 'prediabetes', 'type 2 diabetes', 'hypertension', 'diabetes']
# RISK_FACTOR_VOCAB = ['diabetes', 'hypertension', 'smoking', 'dyslipidemia', 'obesity', 'tobacco use', 'family history', 'chronic kidney disease', 'physical inactivity']
# INTERVENTION_VOCAB = ['statin', 'aspirin', 'ace inhibitor', 'angiotensin receptor blocker', 'arb', 'calcium channel blocker', 'ccb', 'thiazide', 'diuretic', 'metformin', 'sulfonylurea', 'insulin', 'lifestyle counseling', 'behavioral counseling']

# def normalize_entity(name):
#     return re.sub(r'\s+', ' ', str(name).strip().lower())

# def rule_based_extract(text):
#     """Keyword/regex fallback used when GROQ_API_KEY is not configured."""
#     lowered = text.lower()
#     conditions = sorted({c for c in CONDITION_VOCAB if c in lowered})
#     risk_factors = sorted({r for r in RISK_FACTOR_VOCAB if r in lowered})
#     interventions = sorted({i for i in INTERVENTION_VOCAB if i in lowered})
#     population_match = re.search(r'adults?[^.]{0,80}?aged \d{1,2} to \d{1,2} years[^.]{0,80}', text, re.I)
#     population = population_match.group(0).strip() if population_match else None
#     thresholds = sorted(set(re.findall(r'\d{1,3}(?:\.\d+)?%', text)))[:5]
#     grade_match = re.search(r'\((A|B|C|D|I) recommendation\)|\bGrade:\s*(A|B|C|D|I)\b', text)
#     grade = next((g for g in grade_match.groups() if g), None) if grade_match else None
#     if not any([conditions, risk_factors, interventions, population, thresholds]):
#         return {'organization': None, 'recommendations': []}
#     return {
#         'organization': None,
#         'recommendations': [{
#             'text': text.strip()[:280],
#             'grade': grade,
#             'population': population,
#             'conditions': conditions,
#             'risk_factors': risk_factors,
#             'interventions': interventions,
#             'thresholds': thresholds,
#         }],
#     }

# def _parse_json_response(raw):
#     raw = re.sub(r'^```(json)?', '', raw.strip()).strip()
#     raw = re.sub(r'```$', '', raw).strip()
#     return json.loads(raw)

# def llm_extract(text, model=CONFIG['groq_model']):
#     from groq import Groq
#     client = Groq(api_key=GROQ_API_KEY)
#     system_prompt = (
#         'You extract structured facts from a single passage of a clinical practice '
#         'guideline. Respond with JSON only, no prose, no markdown fences. Schema: '
#         '{"organization": string|null, "recommendations": [{"text": string, '
#         '"grade": string|null, "population": string|null, "conditions": [string], '
#         '"risk_factors": [string], "interventions": [string], "thresholds": [string]}]}. '
#         'Only extract facts explicitly stated in the passage. If the passage contains '
#         'no clinical recommendation, return {"organization": null, "recommendations": []}.'
#     )
#     response = client.chat.completions.create(
#         model=model,
#         messages=[
#             {'role': 'system', 'content': system_prompt},
#             {'role': 'user', 'content': text[:3000]},
#         ],
#         temperature=0,
#     )
#     return _parse_json_response(response.choices[0].message.content)

# def extract_entities(text, use_llm=True):
#     if use_llm and GROQ_API_KEY:
#         try:
#             data = llm_extract(text)
#             if isinstance(data, dict) and 'recommendations' in data:
#                 return data
#         except Exception as exc:
#             print(f'  ⚠️ LLM extraction failed ({exc}); falling back to rules for this chunk.')
#     return rule_based_extract(text)

# kg_chunks = chunks_df[chunks_df['section'].isin(CONFIG['kg_source_sections'])].reset_index(drop=True)
# if CONFIG['kg_max_chunks']:
#     kg_chunks = kg_chunks.head(CONFIG['kg_max_chunks'])

# use_llm_extraction = bool(GROQ_API_KEY)
# print(f"Extracting entities from {len(kg_chunks)} chunks "
#       f"({'LLM via Groq' if use_llm_extraction else 'rule-based fallback -- set GROQ_API_KEY for richer extraction'})")

# extraction_records = []
# for _, row in tqdm(kg_chunks.iterrows(), total=len(kg_chunks)):
#     extracted = extract_entities(row['text'], use_llm=use_llm_extraction)
#     extraction_records.append({
#         'chunk_id': row['id'],
#         'source': row['source'],
#         'document': row['document'],
#         'year': row['year'],
#         'page': row['page'],
#         'section': row['section'],
#         'text': row['text'],
#         'organization': extracted.get('organization') or ORGANIZATION_MAP.get(row['source'], row['source']),
#         'recommendations': extracted.get('recommendations', []),
#     })

# with open(ARTIFACT_DIR / 'kg_extractions.json', 'w', encoding='utf-8') as f:
#     json.dump(extraction_records, f, ensure_ascii=False, indent=2)

# kg_extractions_by_chunk = {r['chunk_id']: r['recommendations'] for r in extraction_records}
# n_with_recs = sum(1 for r in extraction_records if r['recommendations'])
# print(f'Chunks processed: {len(extraction_records)} | chunks with ≥1 recommendation: {n_with_recs}')





In [ ]:
!pip install groq json_repair Pillow tqdm nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 4.6 MB/s eta 0:00:00


In [ ]:
#@title 9) Entity and relationship extraction
import asyncio
import json
import re
import os
from tqdm.auto import tqdm
from gemini_webapi import GeminiClient
import json_repair
import nest_asyncio

# ============================================
# 1. إعدادات وقواعد الاستخراج الأساسية
# ============================================
ORGANIZATION_MAP = {'USPSTF': 'US Preventive Services Task Force', 'WHO': 'World Health Organization'}
CONDITION_VOCAB = ['cardiovascular disease', 'coronary heart disease', 'stroke', 'prediabetes', 'type 2 diabetes', 'hypertension', 'diabetes']
RISK_FACTOR_VOCAB = ['diabetes', 'hypertension', 'smoking', 'dyslipidemia', 'obesity', 'tobacco use', 'family history', 'chronic kidney disease', 'physical inactivity']
INTERVENTION_VOCAB = ['statin', 'aspirin', 'ace inhibitor', 'angiotensin receptor blocker', 'arb', 'calcium channel blocker', 'ccb', 'thiazide', 'diuretic', 'metformin', 'sulfonylurea', 'insulin', 'lifestyle counseling', 'behavioral counseling']

def normalize_entity(name):
    return re.sub(r'\s+', ' ', str(name).strip().lower())

def rule_based_extract(text):
    """Keyword/regex fallback used when LLM extraction fails."""
    lowered = text.lower()
    conditions = sorted({c for c in CONDITION_VOCAB if c in lowered})
    risk_factors = sorted({r for r in RISK_FACTOR_VOCAB if r in lowered})
    interventions = sorted({i for i in INTERVENTION_VOCAB if i in lowered})
    population_match = re.search(r'adults?[^.]{0,80}?aged \d{1,2} to \d{1,2} years[^.]{0,80}', text, re.I)
    population = population_match.group(0).strip() if population_match else None
    thresholds = sorted(set(re.findall(r'\d{1,3}(?:\.\d+)?%', text)))[:5]
    grade_match = re.search(r'\((A|B|C|D|I) recommendation\)|\bGrade:\s*(A|B|C|D|I)\b', text)
    grade = next((g for g in grade_match.groups() if g), None) if grade_match else None

    if not any([conditions, risk_factors, interventions, population, thresholds]):
        return {'organization': None, 'recommendations': []}

    return {
        'organization': None,
        'recommendations': [{
            'text': text.strip()[:280],
            'grade': grade,
            'population': population,
            'conditions': conditions,
            'risk_factors': risk_factors,
            'interventions': interventions,
            'thresholds': thresholds,
            'relationships': []  # 💡 تم إضافة مصفوفة العلاقات الفارغة لضمان تطابق الـ Schema
        }],
    }

def _parse_json_response(raw):
    if not raw: return None
    raw = re.sub(r'^```(json)?', '', raw.strip(), flags=re.I).strip()
    raw = re.sub(r'```$', '', raw).strip()
    try:
        return json_repair.loads(raw)
    except:
        return json.loads(raw)

# ============================================
# 2. إعدادات حسابات Gemini (الكوكيز)
# ============================================
ACCOUNTS = [
     {
        "name": "Account_1",
        "psid":"g.a000AwnVrLHfPm0y_mHzkjiXuTP3vjD4aqNIylryUbpgocRisz_1niU6gCZaL-r_-ZJOi-JyTgACgYKARYSARMSFQHGX2MiEP1lcMQ3AcUI8y8yVMmAfhoVAUF8yKpf4zKzWjekXkxLb8Dn_8m60076",
        "psidts": "sidts-CjcBPWEu2foFY8YLdY02z8EPBh2LU56ZkwKFzyF0lwpYI8yDXik6dlVQqtIIPe8IaGeIxxYHDNRPEAA",
        "used_count": 0,
        "hourly_limit": 100,
        "failed_count": 0,
        "last_used": 0
     }
]

class MultiAccountGeminiGenerator:
    def __init__(self, accounts):
        self.accounts = accounts
        self.clients = {}

    async def init_all(self):
        print("\n🔄 تهيئة حسابات Gemini...")
        for acc in self.accounts:
            try:
                client = GeminiClient(acc["psid"], acc["psidts"])
                await client.init(timeout=60, auto_refresh=True)
                self.clients[acc["name"]] = client
                print(f"✅ {acc['name']} - متصل")
            except Exception as e:
                print(f"❌ {acc['name']} - فشل: {str(e)[:50]}")
                acc["hourly_limit"] = 0

    def get_best_account(self):
        available = [acc for acc in self.accounts if acc["hourly_limit"] > 0 and acc["used_count"] < acc["hourly_limit"]]
        if not available:
            return None
        return min(available, key=lambda x: x["used_count"] + (x["failed_count"] * 2))

    async def generate(self, prompt):
        max_retries = 3
        for attempt in range(max_retries):
            acc = self.get_best_account()
            if not acc:
                print(f"\n⏳ تم استنفاد الحد الأقصى للحسابات... انتظار 10 دقائق")
                await asyncio.sleep(600)
                for a in self.accounts: a["used_count"] = 0
                continue

            client = self.clients[acc["name"]]
            try:
                response = await asyncio.wait_for(
                    client.generate_content(prompt, timeout=120),
                    timeout=150
                )
                acc["used_count"] += 1
                acc["failed_count"] = 0
                return response.text
            except Exception as e:
                acc["failed_count"] += 1
                print(f"⚠️ خطأ في الاستخراج: {str(e)[:60]}")
                await asyncio.sleep(5)
        return None

# ============================================
# 3. دالة الاستخراج المتزامنة (مع الشيك بوينت)
# ============================================
async def extract_all_chunks(kg_chunks, output_file):
    generator = MultiAccountGeminiGenerator(ACCOUNTS)
    await generator.init_all()

    extraction_records = []
    processed_ids = set()

    # [CHECKPOINT LOAD] قراءة الملف لو موجود عشان منبدأش من الصفر
    if os.path.exists(output_file):
        try:
            with open(output_file, 'r', encoding='utf-8') as f:
                extraction_records = json.load(f)
            processed_ids = {r['chunk_id'] for r in extraction_records}
            print(f"🔄 [Checkpoint] تم استرجاع {len(extraction_records)} سجل محفوظ مسبقاً.")
        except Exception as e:
            print(f"⚠️ [Checkpoint Error] فشل قراءة الملف السابق: {e}")

    pending_chunks = kg_chunks[~kg_chunks['id'].isin(processed_ids)]

    if pending_chunks.empty:
        print("✅ جميع البيانات تمت معالجتها مسبقاً! لا يوجد شيء جديد.")
        return extraction_records

    print(f"🚀 متبقي {len(pending_chunks)} سجل للمعالجة من أصل {len(kg_chunks)}...")

    # 💡 التعديل هنا: استخدام الـ Prompt الخاص بك الدقيق جداً لمنع الـ Hallucinations
    system_prompt = (
        'You extract structured facts from a single passage of a clinical '
        'practice guideline. Respond with JSON only, no prose, no markdown. '
        'Schema: '
        '{"organization": string|null, '
        '"recommendations": [{'
        '"text": string, '
        '"grade": string|null, '
        '"population": string|null, '
        '"conditions": [string], '
        '"risk_factors": [string], '
        '"interventions": [string], '
        '"thresholds": [string], '
        '"relationships": [{"subject": string, '
        '"relation": string, '
        '"object": string}]'
        '}]} . '
        'Only extract facts explicitly stated in the passage. '
        'Only create a relationship when the relationship is explicitly '
        'supported by the passage. Do not infer medical relationships. '
        'If there is no clinical recommendation, return '
        '{"organization": null, "recommendations": []}.'
    )

    for _, row in tqdm(pending_chunks.iterrows(), total=len(pending_chunks)):
        full_prompt = f"{system_prompt}\n\nPassage to extract from:\n{row['text'][:3000]}"
        extracted_data = None

        llm_response = await generator.generate(full_prompt)

        if llm_response:
            try:
                parsed_data = _parse_json_response(llm_response)
                if isinstance(parsed_data, dict) and 'recommendations' in parsed_data:
                    extracted_data = parsed_data
            except Exception as exc:
                print(f'  ⚠️ LLM parsing failed ({exc}); falling back to rules for this chunk.')

        if not extracted_data:
            extracted_data = rule_based_extract(row['text'])

        extraction_records.append({
            'chunk_id': row['id'],
            'source': row['source'],
            'document': row['document'],
            'year': row['year'],
            'page': row['page'],
            'section': row['section'],
            'text': row['text'],
            'organization': extracted_data.get('organization') or ORGANIZATION_MAP.get(row['source'], row['source']),
            'recommendations': extracted_data.get('recommendations', []),
        })

        # [CHECKPOINT SAVE] حفظ فوري للملف
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(extraction_records, f, ensure_ascii=False, indent=2)

        await asyncio.sleep(2)

    return extraction_records

# ============================================
# 4. نقطة الانطلاق الرئيسية (Main Execution)
# ============================================

# 💡 تأكيد قسري لمعالجة كل البيانات للمشروع النهائي (لا يتم تجاهل أي Chunks)
CONFIG['kg_max_chunks'] = None

kg_chunks = chunks_df[chunks_df['section'].isin(CONFIG['kg_source_sections'])].reset_index(drop=True)
if CONFIG['kg_max_chunks']:
    kg_chunks = kg_chunks.head(CONFIG['kg_max_chunks'])

OUTPUT_FILE_PATH = ARTIFACT_DIR / 'kg_extractions.json'

nest_asyncio.apply()
extraction_records = asyncio.run(extract_all_chunks(kg_chunks, OUTPUT_FILE_PATH))

kg_extractions_by_chunk = {r['chunk_id']: r['recommendations'] for r in extraction_records}
n_with_recs = sum(1 for r in extraction_records if r['recommendations'])
print("\n" + "="*50)
print(f'✅ اكتملت العملية بنجاح!')
print(f'Chunks processed: {len(extraction_records)} | chunks with ≥1 recommendation: {n_with_recs}')
print("="*50)


🔄 تهيئة حسابات Gemini...


2026-08-13 09:38:20.833 | SUCCESS  | gemini_webapi.client:init:270 - Gemini client initialized successfully.


✅ Account_1 - متصل
🔄 [Checkpoint] تم استرجاع 263 سجل محفوظ مسبقاً.
🚀 متبقي 43 سجل للمعالجة من أصل 306...


  0%|          | 0/43 [00:00<?, ?it/s]

2026-08-13 09:39:33.196 | DEBUG    | gemini_webapi.utils.parsing:parse_response_by_frame:188 - Incomplete frame at position 0: expected 16971 UTF-16 units, but received 16323. Waiting for additional data...
2026-08-13 09:39:33.239 | DEBUG    | gemini_webapi.utils.parsing:parse_response_by_frame:188 - Incomplete frame at position 0: expected 16971 UTF-16 units, but received 16323. Waiting for additional data...
2026-08-13 09:39:33.252 | DEBUG    | gemini_webapi.utils.parsing:parse_response_by_frame:188 - Incomplete frame at position 0: expected 16970 UTF-16 units, but received 16323. Waiting for additional data...
2026-08-13 09:42:51.908 | DEBUG    | gemini_webapi.utils.parsing:parse_response_by_frame:188 - Incomplete frame at position 0: expected 16640 UTF-16 units, but received 16295. Waiting for additional data...
2026-08-13 09:42:52.084 | DEBUG    | gemini_webapi.utils.parsing:parse_response_by_frame:188 - Incomplete frame at position 0: expected 17064 UTF-16 units, but received 162


✅ اكتملت العملية بنجاح!
Chunks processed: 306 | chunks with ≥1 recommendation: 141


In [ ]:
#@title 10) Build the Neo4j Knowledge Graph

import hashlib
import pandas as pd


def write_knowledge_graph(records):

    # =========================================================
    # 1. Evidence rows
    # =========================================================

    evidence_rows = []

    for r in records:
        evidence_rows.append({
            'organization': r.get('organization'),
            'source': r.get('source'),
            'document': r.get('document'),
            'year': r.get('year'),
            'chunk_id': r.get('chunk_id'),
            'page': r.get('page'),
            'section': r.get('section'),
            'text': r.get('text'),
        })


    # =========================================================
    # 2. Recommendation rows
    # =========================================================

    recommendation_rows = []

    for r in records:

        for i, rec in enumerate(r.get('recommendations', [])):

            recommendation_id = hashlib.sha256(
                f"{r['chunk_id']}|{i}".encode()
            ).hexdigest()[:20]

            recommendation_rows.append({

                'source': r.get('source'),
                'document': r.get('document'),
                'chunk_id': r.get('chunk_id'),

                'recommendation_id': recommendation_id,

                'text': rec.get('text'),
                'grade': rec.get('grade'),

                'population': (
                    normalize_entity(rec['population'])
                    if rec.get('population')
                    else None
                ),

                'conditions': [
                    normalize_entity(x)
                    for x in rec.get('conditions', [])
                    if x
                ],

                'risk_factors': [
                    normalize_entity(x)
                    for x in rec.get('risk_factors', [])
                    if x
                ],

                'interventions': [
                    normalize_entity(x)
                    for x in rec.get('interventions', [])
                    if x
                ],

                'thresholds': [
                    str(x).strip()
                    for x in rec.get('thresholds', [])
                    if x
                ],

                # IMPORTANT:
                # relationships extracted by Gemini
                'relationships': [
                    {
                        'subject': normalize_entity(rel.get('subject')),
                        'relation': normalize_entity(rel.get('relation')),
                        'object': normalize_entity(rel.get('object'))
                    }
                    for rel in rec.get('relationships', [])
                    if (
                        rel.get('subject')
                        and rel.get('relation')
                        and rel.get('object')
                    )
                ]
            })


    # =========================================================
    # 3. Evidence graph
    # =========================================================

    evidence_cypher = '''
    UNWIND $rows AS row

    MERGE (o:Organization {name: row.organization})

    MERGE (g:Guideline {
        source: row.source,
        document: row.document
    })

    ON CREATE SET g.year = row.year

    MERGE (o)-[:PUBLISHES]->(g)

    MERGE (e:Evidence {
        chunk_id: row.chunk_id
    })

    SET
        e.source = row.source,
        e.document = row.document,
        e.page = row.page,
        e.section = row.section,
        e.text = row.text

    MERGE (g)-[:CONTAINS]->(e)
    '''


    # =========================================================
    # 4. Recommendation + entities
    # =========================================================

    recommendation_cypher = '''

    UNWIND $rows AS row

    MATCH (e:Evidence {
        chunk_id: row.chunk_id
    })

    MATCH (g:Guideline {
        source: row.source,
        document: row.document
    })

    MERGE (r:Recommendation {
        id: row.recommendation_id
    })

    SET
        r.text = row.text,
        r.grade = row.grade

    MERGE (g)-[:HAS_RECOMMENDATION]->(r)

    MERGE (r)-[:SUPPORTED_BY]->(e)


    // Population
    FOREACH (
        pop IN CASE
            WHEN row.population IS NULL
            THEN []
            ELSE [row.population]
        END |

        MERGE (p:Population {
            description: pop
        })

        MERGE (r)-[:APPLIES_TO]->(p)
    )


    // Conditions
    FOREACH (
        cond IN row.conditions |

        MERGE (c:Condition {
            name: cond
        })

        MERGE (r)-[:FOR_CONDITION]->(c)
    )


    // Risk factors
    FOREACH (
        rf IN row.risk_factors |

        MERGE (rfN:RiskFactor {
            name: rf
        })

        MERGE (r)-[:CONSIDERS_RISK_FACTOR]->(rfN)
    )


    // Interventions
    FOREACH (
        iv IN row.interventions |

        MERGE (ivN:Intervention {
            name: iv
        })

        MERGE (r)-[:SUGGESTS_INTERVENTION]->(ivN)
    )


    // Thresholds
    FOREACH (
        th IN row.thresholds |

        MERGE (thN:Threshold {
            description: th
        })

        MERGE (r)-[:HAS_THRESHOLD]->(thN)
    )
    '''


    # =========================================================
    # 5. Gemini LLM relationships
    # =========================================================

    relationship_cypher = '''
    UNWIND $rows AS row

    MATCH (r:Recommendation {
        id: row.recommendation_id
    })

    UNWIND row.relationships AS rel

    MERGE (s:Entity {
        name: rel.subject
    })

    MERGE (o:Entity {
        name: rel.object
    })

    MERGE (s)-[relationship:LLM_RELATION]->(o)

    SET
        relationship.predicate = rel.relation,
        relationship.source = row.source,
        relationship.document = row.document,
        relationship.chunk_id = row.chunk_id,
        relationship.recommendation_id = row.recommendation_id
    '''


    # =========================================================
    # 6. Write to Neo4j
    # =========================================================

    with neo4j_driver.session(
        database=NEO4J_DATABASE
    ) as session:

        # Evidence
        session.run(
            evidence_cypher,
            rows=evidence_rows
        )

        # Recommendations + entities
        if recommendation_rows:

            session.run(
                recommendation_cypher,
                rows=recommendation_rows
            )

            # LLM relationships
            session.run(
                relationship_cypher,
                rows=recommendation_rows
            )


    return {
        'evidence_nodes_written': len(evidence_rows),
        'recommendation_nodes_written': len(recommendation_rows),
        'relationship_records_processed': sum(
            len(r.get('relationships', []))
            for r in recommendation_rows
        )
    }


# =========================================================
# 7. Execute
# =========================================================

kg_write_summary = write_knowledge_graph(
    extraction_records
)

print("\n" + "=" * 70)
print("✅ KNOWLEDGE GRAPH BUILD COMPLETE")
print("=" * 70)

print(
    "Knowledge graph write summary:",
    kg_write_summary
)


# =========================================================
# 8. Count nodes
# =========================================================

with neo4j_driver.session(
    database=NEO4J_DATABASE
) as session:

    counts = session.run('''
        MATCH (n)
        UNWIND labels(n) AS label
        RETURN label, count(*) AS n
        ORDER BY n DESC
    ''')

    counts_df = pd.DataFrame(
        [dict(r) for r in counts]
    )

print("\n📊 Node counts:")
print(
    counts_df.to_string(index=False)
)


✅ KNOWLEDGE GRAPH BUILD COMPLETE
Knowledge graph write summary: {'evidence_nodes_written': 306, 'recommendation_nodes_written': 511, 'relationship_records_processed': 448}

📊 Node counts:
           label   n
  Recommendation 511
    Intervention 472
          Entity 398
        Evidence 306
       Threshold 270
       Condition 166
      Population 154
      RiskFactor 110
    Organization  14
  GuidelineChunk  10
       Guideline   9
ClinicalQuestion   1


In [ ]:
#@title 11) Vector retrieval engine

def retrieve(
    query,
    top_k=CONFIG['default_top_k'],
    source=None
):

    # =========================================================
    # 1. Encode user query
    # =========================================================

    query_embedding = embedder.encode(
        [f"query: {query}"],
        normalize_embeddings=True
    ).tolist()


    # =========================================================
    # 2. Optional source filter
    # =========================================================

    where = (
        {"source": source}
        if source
        else None
    )


    # =========================================================
    # 3. ChromaDB similarity search
    # =========================================================

    result = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where=where,
        include=[
            "documents",
            "metadatas",
            "distances"
        ],
    )


    # =========================================================
    # 4. Convert results to DataFrame
    # =========================================================

    rows = []

    documents = result.get(
        "documents",
        [[]]
    )[0]

    metadatas = result.get(
        "metadatas",
        [[]]
    )[0]

    distances = result.get(
        "distances",
        [[]]
    )[0]

    ids = result.get(
        "ids",
        [[]]
    )[0]


    for text, metadata, distance, chunk_id in zip(
        documents,
        metadatas,
        distances,
        ids
    ):

        metadata = metadata or {}

        rows.append({

            "id": chunk_id,

            "text": text,

            "distance": float(
                distance
            ),

            **metadata
        })


    return pd.DataFrame(rows)


# =========================================================
# 5. Test retrieval
# =========================================================

query = (
    "58-year-old patient with diabetes "
    "and hypertension. Should statin therapy "
    "be considered?"
)

evidence = retrieve(
    query,
    top_k=10
)


# =========================================================
# 6. Display retrieved evidence
# =========================================================

display(
    evidence[
        [
            "source",
            "document",
            "page",
            "section",
            "distance",
            "text"
        ]
    ]
)

,source,document,page,section,distance,text
0,USPSTF,01_statins_2022.pdf,8,Recommendation,0.226122,ravastatin vs usual care: the Antihypertensive...
1,USPSTF,01_statins_2022.pdf,7,Recommendation,0.245096,"ntonio LD, Brown TM, Carson AP, Safford MM. Ra..."
2,USPSTF,01_statins_2022.pdf,8,Recommendation,0.254324,"18. Wall HK, Ritchey MD, Gillespie C, Omura JD..."
3,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,43,Evidence Assessment,0.259233,"13. James PA, Oparil S, Carter BL, et al. 2014..."
4,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,39,Evidence Assessment,0.260840,•• Aged under 40 years with BP ≥140/90 mmHg (t...
5,USPSTF,01_statins_2022.pdf,4,Recommendation,0.266263,compared with placebo.13 Almost all trials did...
6,USPSTF,01_statins_2022.pdf,6,Recommendation,0.267139,tcomes are also needed. • Definitivelydetermin...
7,USPSTF,01_statins_2022.pdf,7,Recommendation,0.273247,Equations and inform decision-making for perso...
8,USPSTF,01_statins_2022.pdf,4,Recommendation,0.273424,re additional tools and resources? CVD is the ...
9,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,43,Evidence Assessment,0.278481,tors. Effects of an angiotensin-converting–enz...


In [ ]:
#@title 12) Graph retrieval engine

QUESTION_ENTITY_VOCAB = sorted(
    set(CONDITION_VOCAB)
    | set(RISK_FACTOR_VOCAB)
    | set(INTERVENTION_VOCAB)
)


# =========================================================
# 1. Link entities from the clinical question
# =========================================================

def link_question_entities(question):

    q = question.lower()

    return [
        entity
        for entity in QUESTION_ENTITY_VOCAB
        if entity.lower() in q
    ]


# =========================================================
# 2. Graph retrieval
# =========================================================

def graph_retrieve(
    question,
    limit=CONFIG['graph_retrieval_limit']
):

    entities = link_question_entities(question)

    columns = [
        'id',
        'source',
        'document',
        'page',
        'section',
        'text',
        'recommendation_text',
        'grade',
        'matched_entity'
    ]

    if not entities:
        return pd.DataFrame(columns=columns)


    # =====================================================
    # Match Conditions / Risk Factors / Interventions
    # =====================================================

    cypher = '''

    MATCH (n)
    WHERE
        (n:Condition OR
         n:RiskFactor OR
         n:Intervention)
        AND toLower(n.name) IN $entities

    MATCH (
        n
    )<-[
        :FOR_CONDITION |
        :CONSIDERS_RISK_FACTOR |
        :SUGGESTS_INTERVENTION
    ]-(r:Recommendation)

    MATCH (r)-[:SUPPORTED_BY]->(e:Evidence)

    RETURN DISTINCT

        e.chunk_id AS id,

        e.source AS source,

        e.document AS document,

        e.page AS page,

        e.section AS section,

        e.text AS text,

        r.text AS recommendation_text,

        r.grade AS grade,

        n.name AS matched_entity

    LIMIT $limit
    '''


    with neo4j_driver.session(
        database=NEO4J_DATABASE
    ) as session:

        rows = [
            dict(record)
            for record in session.run(
                cypher,
                entities=entities,
                limit=limit
            )
        ]


    if not rows:
        return pd.DataFrame(
            columns=columns
        )


    return pd.DataFrame(
        rows,
        columns=columns
    )


# =========================================================
# 3. Test
# =========================================================

print(
    "Entities linked from demo question:"
)

print(
    link_question_entities(query)
)


graph_evidence = graph_retrieve(
    query
)


display(
    graph_evidence[
        [
            'source',
            'document',
            'page',
            'matched_entity',
            'grade',
            'recommendation_text'
        ]
    ]
)

Entities linked from demo question:
['diabetes', 'hypertension', 'statin']


,source,document,page,matched_entity,grade,recommendation_text
0,USPSTF,01_statins_2022.pdf,1,statin,B,The USPSTF recommends that clinicians prescrib...
1,USPSTF,01_statins_2022.pdf,1,statin,None,The USPSTF recommends that clinicians selectiv...
2,USPSTF,01_statins_2022.pdf,1,statin,C,The USPSTF recommends that clinicians selectiv...
3,USPSTF,01_statins_2022.pdf,2,statin,B,The USPSTF recommends that clinicians prescrib...
4,USPSTF,01_statins_2022.pdf,2,statin,C,The USPSTF recommends that clinicians selectiv...
5,USPSTF,01_statins_2022.pdf,2,statin,I,The USPSTF concludes that the current evidence...
6,USPSTF,01_statins_2022.pdf,2,statin,None,is insufficient to assess the balance of benef...
7,USPSTF,01_statins_2022.pdf,3,statin,None,Clinicians prescribe a statin once the rationa...
8,USPSTF,01_statins_2022.pdf,3,statin,None,For patients with an estimated 10-year CVD ris...
9,USPSTF,01_statins_2022.pdf,4,statin,None,In patients who have a cardiovascular risk fac...


In [ ]:
#@title 13) Hybrid retrieval (Vector + Graph)

def hybrid_retrieve(
    question,
    top_k=CONFIG['default_top_k'],
    graph_limit=CONFIG['graph_retrieval_limit']
):

    # =========================================================
    # 1. Vector retrieval
    # =========================================================

    vector_hits = retrieve(
        question,
        top_k=top_k
    )


    # =========================================================
    # 2. Graph retrieval
    # =========================================================

    graph_hits = graph_retrieve(
        question,
        limit=graph_limit
    )


    # =========================================================
    # 3. Combine results by chunk ID
    # =========================================================

    combined = {}


    # -------------------------
    # Vector results
    # -------------------------

    for _, row in vector_hits.iterrows():

        record = row.to_dict()

        record['origin'] = 'vector'

        combined[record['id']] = record


    # -------------------------
    # Graph results
    # -------------------------

    for _, row in graph_hits.iterrows():

        record = row.to_dict()

        chunk_id = record['id']


        if chunk_id in combined:

            # Retrieved by BOTH systems
            combined[chunk_id]['origin'] = 'both'


            # Preserve Graph information
            if record.get('recommendation_text') is not None:
                combined[chunk_id][
                    'recommendation_text'
                ] = record[
                    'recommendation_text'
                ]


            if record.get('grade') is not None:
                combined[chunk_id][
                    'grade'
                ] = record['grade']


            if record.get('matched_entity') is not None:
                combined[chunk_id][
                    'matched_entity'
                ] = record['matched_entity']


        else:

            # Graph-only result
            record['origin'] = 'graph'

            record['distance'] = np.nan

            combined[chunk_id] = record


    # =========================================================
    # 4. DataFrame
    # =========================================================

    if not combined:
        return pd.DataFrame()


    return pd.DataFrame(
        list(combined.values())
    )


# =========================================================
# 5. Test
# =========================================================

hybrid_hits = hybrid_retrieve(
    query
)


# =========================================================
# 6. Statistics
# =========================================================

if not hybrid_hits.empty:

    print(
        f"Vector-only: "
        f"{(hybrid_hits['origin'] == 'vector').sum()} | "
        f"Graph-only: "
        f"{(hybrid_hits['origin'] == 'graph').sum()} | "
        f"Both: "
        f"{(hybrid_hits['origin'] == 'both').sum()}"
    )


# =========================================================
# 7. Display
# =========================================================

display(
    hybrid_hits[
        [
            'id',
            'source',
            'document',
            'page',
            'origin',
            'matched_entity'
        ]
    ]
)

Vector-only: 9 | Graph-only: 6 | Both: 4


,id,source,document,page,origin,matched_entity
0,32d7abc608a482bc0d41,USPSTF,01_statins_2022.pdf,8,vector,NaN
1,04582129ce7934dfec8d,USPSTF,01_statins_2022.pdf,7,vector,NaN
2,7531c82d3e790ad6159f,USPSTF,01_statins_2022.pdf,8,vector,NaN
3,c785160d54c705521e28,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,43,vector,NaN
4,41df278bb4c02c8db72d,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,39,vector,NaN
5,3996cba76e45cdd8f510,USPSTF,01_statins_2022.pdf,4,vector,NaN
6,f2c416dea95b14295604,USPSTF,01_statins_2022.pdf,6,vector,NaN
7,1c0e21616c4b8788a784,USPSTF,01_statins_2022.pdf,7,vector,NaN
8,a2ee7a64266a5d6ced52,USPSTF,01_statins_2022.pdf,4,both,statin
9,76882b1141e65d99bef3,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,43,vector,NaN


In [ ]:
#@title 14) Medical Evidence Reranking

from sentence_transformers import CrossEncoder
import pandas as pd
import numpy as np


# =========================================================
# 1. Load reranker
# =========================================================

RERANKER_MODEL = CONFIG['reranker_model']

print(f"Loading reranker: {RERANKER_MODEL}")

reranker = CrossEncoder(
    RERANKER_MODEL
)


# =========================================================
# 2. Build evidence text for reranking
# =========================================================

def build_rerank_text(row):

    parts = []

    # Main evidence
    text = row.get('text')

    if pd.notna(text) and text:
        parts.append(
            str(text)
        )

    # Graph-derived recommendation
    recommendation = row.get(
        'recommendation_text'
    )

    if pd.notna(recommendation) and recommendation:
        parts.append(
            f"Recommendation: {recommendation}"
        )

    # Recommendation grade
    grade = row.get('grade')

    if pd.notna(grade) and grade:
        parts.append(
            f"Grade: {grade}"
        )

    # Entity matched through graph
    entity = row.get(
        'matched_entity'
    )

    if pd.notna(entity) and entity:
        parts.append(
            f"Clinical entity: {entity}"
        )

    return "\n".join(parts)


# =========================================================
# 3. Reranking
# =========================================================

def rerank(
    question,
    hits_df,
    top_n=CONFIG['rerank_top_n']
):

    if (
        hits_df is None
        or hits_df.empty
    ):
        return hits_df


    hits_df = hits_df.copy()


    # -----------------------------------------------------
    # Build passages
    # -----------------------------------------------------

    passages = [
        build_rerank_text(row)
        for _, row in hits_df.iterrows()
    ]


    # -----------------------------------------------------
    # Create question-passage pairs
    # -----------------------------------------------------

    pairs = [
        (question, passage)
        for passage in passages
    ]


    # -----------------------------------------------------
    # Cross-encoder scoring
    # -----------------------------------------------------

    scores = reranker.predict(
        pairs,
        show_progress_bar=False
    )


    hits_df['rerank_score'] = [
        float(score)
        for score in scores
    ]


    # -----------------------------------------------------
    # Sort by relevance
    # -----------------------------------------------------

    hits_df = hits_df.sort_values(
        'rerank_score',
        ascending=False
    )


    # -----------------------------------------------------
    # Remove duplicate chunks
    # -----------------------------------------------------

    hits_df = hits_df.drop_duplicates(
        subset=['id'],
        keep='first'
    )


    # -----------------------------------------------------
    # Keep top N
    # -----------------------------------------------------

    hits_df = hits_df.head(
        top_n
    ).reset_index(drop=True)


    return hits_df


# =========================================================
# 4. Run reranking
# =========================================================

reranked_hits = rerank(
    query,
    hybrid_hits
)


# =========================================================
# 5. Display results
# =========================================================

if not reranked_hits.empty:

    display(
        reranked_hits[
            [
                'id',
                'source',
                'document',
                'page',
                'origin',
                'rerank_score'
            ]
        ]
    )

else:

    print(
        "⚠️ No evidence available for reranking."
    )

Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

,id,source,document,page,origin,rerank_score
0,a4335419d24c0eb9982c,USPSTF,01_statins_2022.pdf,3,both,2.105894
1,3996cba76e45cdd8f510,USPSTF,01_statins_2022.pdf,4,vector,1.404370
2,e5b9c380809b9fff46e3,WHO,HEARTS_Evidence-based_treatment_protocols.pdf,14,graph,1.367043
3,dfe3038c543f8cc58cbe,USPSTF,01_statins_2022.pdf,2,both,1.296975
4,af5972fbfc4119a96a19,USPSTF,01_statins_2022.pdf,1,both,1.265875
5,a2ee7a64266a5d6ced52,USPSTF,01_statins_2022.pdf,4,both,-1.188477
6,3d15881d10d55388b303,USPSTF,01_statins_2022.pdf,1,graph,-1.535466
7,f2c416dea95b14295604,USPSTF,01_statins_2022.pdf,6,vector,-1.798412
8,51df3a379ee163154fe1,USPSTF,01_statins_2022.pdf,2,graph,-1.903128
9,7531c82d3e790ad6159f,USPSTF,01_statins_2022.pdf,8,vector,-2.016364


In [ ]:
#@title 15) Build a grounded evidence packet

def evidence_packet(question, hits):

    # =========================================================
    # No evidence
    # =========================================================

    if hits is None or hits.empty:

        return {
            'status': 'INSUFFICIENT_EVIDENCE',
            'question': question,
            'evidence': []
        }


    evidence_rows = []


    # =========================================================
    # Build evidence records
    # =========================================================

    for _, hit in hits.iterrows():

        # Safe page handling
        page_value = hit.get('page')

        try:
            page_value = int(page_value)
        except:
            page_value = None


        # Safe distance
        distance = hit.get('distance')

        if pd.isna(distance):
            distance = None
        else:
            distance = round(
                float(distance),
                4
            )


        # Safe rerank score
        rerank_score = hit.get(
            'rerank_score'
        )

        if pd.isna(rerank_score):
            rerank_score = None
        else:
            rerank_score = round(
                float(rerank_score),
                4
            )


        evidence_rows.append({

            # ---------------------------------------------
            # Provenance
            # ---------------------------------------------

            'chunk_id': hit.get('id'),

            'source': hit.get(
                'source'
            ),

            'document': hit.get(
                'document'
            ),

            'page': page_value,

            'section': hit.get(
                'section'
            ),


            # ---------------------------------------------
            # Retrieval information
            # ---------------------------------------------

            'origin': hit.get(
                'origin',
                'vector'
            ),

            'distance': distance,

            'rerank_score': rerank_score,


            # ---------------------------------------------
            # Graph information
            # ---------------------------------------------

            'matched_entity': hit.get(
                'matched_entity'
            ),

            'recommendation_text': hit.get(
                'recommendation_text'
            ),

            'grade': hit.get(
                'grade'
            ),


            # ---------------------------------------------
            # Original evidence
            # ---------------------------------------------

            'text': hit.get(
                'text',
                ''
            )
        })


    # =========================================================
    # Evidence packet
    # =========================================================

    return {

        'status': 'SUPPORTED_CANDIDATE',

        'question': question,

        'evidence_count': len(
            evidence_rows
        ),

        'evidence': evidence_rows
    }


# =========================================================
# Test
# =========================================================

packet = evidence_packet(
    query,
    reranked_hits
)


print(
    json.dumps(
        packet,
        ensure_ascii=False,
        indent=2
    )[:12000]
)

{
  "status": "SUPPORTED_CANDIDATE",
  "question": "58-year-old patient with diabetes and hypertension. Should statin therapy be considered?",
  "evidence_count": 10,
  "evidence": [
    {
      "chunk_id": "a4335419d24c0eb9982c",
      "source": "USPSTF",
      "document": "01_statins_2022.pdf",
      "page": 3,
      "section": "Recommendation",
      "origin": "both",
      "distance": null,
      "rerank_score": 2.1059,
      "matched_entity": "statin",
      "recommendation_text": "For patients with an estimated 10-year CVD risk of 7.5% to less than 10% (and who have 1 or more risk factors), clinicians may selectively offer a statin, taking patient values and preferences into account.",
      "grade": null,
      "text": "dyslipidemia, diabetes, or hypertension, the USPSTF recommends that clinicians prescribe a statin once the rationale has been explained and the patient agrees to take a statin.12 For patients with an estimated 10-year CVD risk of 7.5% to less than 10% (and who ha

In [ ]:
#@title 16) Grounded clinical answer generation

import json
import re


def generate_grounded_answer(
    question,
    evidence_records,
    model=CONFIG.get(
        'llm_model',
        'gemini'
    )
):

    # =========================================================
    # 1. Evidence check
    # =========================================================

    if not evidence_records:

        return {
            'status': 'INSUFFICIENT_EVIDENCE',
            'answer': (
                'Insufficient evidence was retrieved '
                'from the available clinical guidelines.'
            ),
            'citations': []
        }


    # =========================================================
    # 2. Build grounded context
    # =========================================================

    context_parts = []

    for i, r in enumerate(
        evidence_records,
        start=1
    ):

        source = r.get(
            'source',
            'Unknown source'
        )

        document = r.get(
            'document',
            'Unknown document'
        )

        page = r.get(
            'page'
        )

        section = r.get(
            'section'
        )

        recommendation = r.get(
            'recommendation_text'
        )

        grade = r.get(
            'grade'
        )

        matched_entity = r.get(
            'matched_entity'
        )

        origin = r.get(
            'origin',
            'unknown'
        )

        text = r.get(
            'text',
            ''
        )


        context_parts.append(
            f"""
[EVIDENCE {i}]
Source: {source}
Document: {document}
Page: {page}
Section: {section}
Retrieved by: {origin}
Matched entity: {matched_entity}
Recommendation: {recommendation}
Grade: {grade}

Passage:
{text}
""".strip()
        )


    context = "\n\n".join(
        context_parts
    )


    # =========================================================
    # 3. Grounding rules
    # =========================================================

    system_prompt = """
You are the reasoning component of a clinical
guideline decision-support system.

You MUST answer ONLY from the supplied evidence.

IMPORTANT RULES:

1. Do not use outside medical knowledge.
2. Do not invent facts, thresholds, diagnoses,
   treatments, or recommendations.
3. Every clinical claim must have one or more
   evidence citations such as [EVIDENCE 1].
4. Preserve the source organization.
5. If different guidelines provide different
   recommendations, explicitly state:

   "GUIDELINES DISAGREE"

6. If the evidence is insufficient, explicitly state:

   "INSUFFICIENT EVIDENCE"

7. Do not resolve a disagreement by guessing.
8. Do not create a recommendation that is not
   explicitly supported by the evidence.
9. Distinguish between evidence from different
   organizations.
10. Keep the answer concise and clinically structured.

Return the answer using this structure:

Clinical interpretation:
...

Guideline evidence:
- Source: ...
- Recommendation: ...
- Evidence: [EVIDENCE X]

Comparison:
...

Conclusion:
...

Evidence confidence:
HIGH / MODERATE / LOW / INSUFFICIENT
"""


    user_prompt = f"""
Question:

{question}

Retrieved clinical evidence:

{context}

Now produce a grounded answer.
"""


    # =========================================================
    # 4. Return prompt package
    #
    # The actual Gemini call can be connected here.
    # =========================================================

    return {
        'status': 'READY_FOR_GROUNDED_GENERATION',
        'question': question,
        'model': model,
        'system_prompt': system_prompt.strip(),
        'user_prompt': user_prompt.strip(),
        'evidence_count': len(
            evidence_records
        )
    }


# =========================================================
# 5. Build grounded answer request
# =========================================================

grounded_request = generate_grounded_answer(
    query,
    packet['evidence']
)


print(
    json.dumps(
        grounded_request,
        ensure_ascii=False,
        indent=2
    )[:15000]
)

{
  "status": "READY_FOR_GROUNDED_GENERATION",
  "question": "58-year-old patient with diabetes and hypertension. Should statin therapy be considered?",
  "model": "gemini",
  "system_prompt": "You are the reasoning component of a clinical\nguideline decision-support system.\n\nYou MUST answer ONLY from the supplied evidence.\n\nIMPORTANT RULES:\n\n1. Do not use outside medical knowledge.\n2. Do not invent facts, thresholds, diagnoses,\n   treatments, or recommendations.\n3. Every clinical claim must have one or more\n   evidence citations such as [EVIDENCE 1].\n4. Preserve the source organization.\n5. If different guidelines provide different\n   recommendations, explicitly state:\n\n   \"GUIDELINES DISAGREE\"\n\n6. If the evidence is insufficient, explicitly state:\n\n   \"INSUFFICIENT EVIDENCE\"\n\n7. Do not resolve a disagreement by guessing.\n8. Do not create a recommendation that is not\n   explicitly supported by the evidence.\n9. Distinguish between evidence from different\n   

In [ ]:
#@title 17) Evidence-only citation verification and safe response builder
SUPPORTED_TERMS = {'statin', 'hypertension', 'diabetes', 'prediabetes', 'aspirin', 'cardiovascular', 'risk', 'lifestyle', 'physical activity', 'diet'}

POSITIVE_STANCE = re.compile(r'\b(recommend(s|ed)?|should be (offered|prescribed|considered)|prescribe)\b', re.I)
CAUTION_STANCE = re.compile(r'\b(insufficient evidence|does not recommend|not recommended|contraindicated|no benefit)\b', re.I)

def normalize_tokens(text):
    return set(re.findall(r'[a-zA-Z][a-zA-Z-]{2,}', text.lower()))

def citation_supports_claim(claim, evidence_text, min_overlap=2):
    claim_tokens = normalize_tokens(claim) & SUPPORTED_TERMS
    evidence_tokens = normalize_tokens(evidence_text)
    return len(claim_tokens & evidence_tokens) >= min_overlap

def sources_conflict(evidence_records):
    """Flag a real disagreement only when sources use opposing language, or
    when the graph-extracted recommendation grade differs in kind (e.g. one
    source has an actionable grade A/B/C and another has grade I --
    insufficient evidence) -- not merely because more than one source
    matched."""
    stance_by_source = {}
    for record in evidence_records:
        text = record['text']
        chunk_id = record.get('chunk_id') or record.get('id')
        source = record['source']
        if CAUTION_STANCE.search(text):
            stance_by_source.setdefault(source, set()).add('caution')
        elif POSITIVE_STANCE.search(text):
            stance_by_source.setdefault(source, set()).add('supportive')
        for rec in kg_extractions_by_chunk.get(chunk_id, []):
            grade = (rec.get('grade') or '').upper()
            if grade == 'I':
                stance_by_source.setdefault(source, set()).add('caution')
            elif grade in {'A', 'B', 'C'}:
                stance_by_source.setdefault(source, set()).add('supportive')
    stances = {stance for stance_set in stance_by_source.values() for stance in stance_set}
    return {'supportive', 'caution'} <= stances

def build_safe_response(question, hits, min_hits=CONFIG['min_supporting_hits']):
    if hits is None or hits.empty:
        return {'status': 'OUT_OF_SCOPE', 'recommendation': 'The available guideline corpus does not provide sufficient evidence for this question.', 'citations': [], 'evidence': []}

    hits = hits.copy()
    hits['supports_query'] = hits['text'].map(lambda t: citation_supports_claim(question, t, min_overlap=1))
    supported = hits[hits['supports_query']].head(6)
    if len(supported) < min_hits:
        return {'status': 'INSUFFICIENT_EVIDENCE', 'recommendation': 'Evidence is insufficient for a grounded answer from the available corpus.', 'citations': [], 'evidence': []}

    evidence_records = supported[['id', 'source', 'document', 'page', 'section', 'text']].rename(columns={'id': 'chunk_id'}).to_dict('records')
    sources = sorted({r['source'] for r in evidence_records})
    status = 'GUIDELINE_DIFFERENCE_DETECTED' if len(sources) > 1 and sources_conflict(evidence_records) else 'SUPPORTED'
    citations = [f"{r['source']} — {r['document']}, p. {int(r['page'])}" for r in evidence_records]
    return {
        'status': status,
        'recommendation': 'Review the cited guideline passages and apply clinical judgment; this system does not make an individualized treatment decision.',
        'citations': citations,
        'evidence': evidence_records,
    }

safe_response = build_safe_response(query, reranked_hits)
print(json.dumps(safe_response, ensure_ascii=False, indent=2)[:12000])


{
  "status": "GUIDELINE_DIFFERENCE_DETECTED",
  "recommendation": "Review the cited guideline passages and apply clinical judgment; this system does not make an individualized treatment decision.",
  "citations": [
    "USPSTF — 01_statins_2022.pdf, p. 3",
    "USPSTF — 01_statins_2022.pdf, p. 4",
    "WHO — HEARTS_Evidence-based_treatment_protocols.pdf, p. 14",
    "USPSTF — 01_statins_2022.pdf, p. 2",
    "USPSTF — 01_statins_2022.pdf, p. 1",
    "USPSTF — 01_statins_2022.pdf, p. 4"
  ],
  "evidence": [
    {
      "chunk_id": "a4335419d24c0eb9982c",
      "source": "USPSTF",
      "document": "01_statins_2022.pdf",
      "page": 3,
      "section": "Recommendation",
      "text": "dyslipidemia, diabetes, or hypertension, the USPSTF recommends that clinicians prescribe a statin once the rationale has been explained and the patient agrees to take a statin.12 For patients with an estimated 10-year CVD risk of 7.5% to less than 10% (and who have \u00021 of the risk factors noted above)

In [ ]:
#@title 18) Store the retrieval trace in Neo4j
def write_retrieval_trace(question, packet):
    cypher = '''
    MERGE (q:ClinicalQuestion {question: $question})
    SET q.created_at = coalesce(q.created_at, datetime())
    WITH q
    UNWIND $evidence AS ev
      MERGE (e:Evidence {chunk_id: ev.chunk_id})
        SET e.source = ev.source, e.document = ev.document, e.page = ev.page,
            e.section = ev.section, e.text = ev.text
      MERGE (q)-[r:RETRIEVED]->(e)
        SET r.origin = ev.origin, r.distance = ev.distance, r.rerank_score = ev.rerank_score
    RETURN count(*) AS relationships
    '''
    with neo4j_driver.session(database=NEO4J_DATABASE) as session:
        return session.run(cypher, question=question, evidence=packet['evidence']).single()['relationships']

if packet['evidence']:
    print('Neo4j RETRIEVED relationships written:', write_retrieval_trace(query, packet))
else:
    print('No evidence to write')


Neo4j RETRIEVED relationships written: 10


In [ ]:
#@title 19) Optional Supabase logging
# Create this table in the Supabase SQL editor first if you want logging:
# create table if not exists retrieval_runs (
#   id bigint generated by default as identity primary key,
#   question text not null,
#   status text not null,
#   evidence jsonb not null,
#   created_at timestamptz default now()
# );

def log_to_supabase(question, packet):
    if supabase is None:
        return 'Supabase skipped'
    payload = {'question': question, 'status': packet['status'], 'evidence': packet['evidence']}
    response = supabase.table('retrieval_runs').insert(payload).execute()
    return response.data

# Uncomment after creating the table and confirming RLS policies:
# print(log_to_supabase(query, packet))


In [ ]:
#@title 20) Compare USPSTF and WHO evidence (vector-only diagnostic)
def compare_sources(question, top_k_each=5):
    us = retrieve(question, top_k=top_k_each, source='USPSTF')
    who = retrieve(question, top_k=top_k_each, source='WHO')
    rows = []
    for label, frame in [('USPSTF', us), ('WHO', who)]:
        for _, r in frame.iterrows():
            rows.append({'source': label, 'document': r['document'], 'page': int(r['page']), 'section': r['section'], 'distance': float(r['distance']), 'text': r['text']})
    return pd.DataFrame(rows)

comparison = compare_sources(query)
print(comparison[['source', 'document', 'page', 'section', 'distance']].to_string(index=False))
print('\nSources retrieved:', sorted(comparison['source'].unique()) if not comparison.empty else [])


source                                      document  page             section  distance
USPSTF                           01_statins_2022.pdf     8      Recommendation  0.226122
USPSTF                           01_statins_2022.pdf     7      Recommendation  0.245096
USPSTF                           01_statins_2022.pdf     8      Recommendation  0.254324
USPSTF                           01_statins_2022.pdf     4      Recommendation  0.266263
USPSTF                           01_statins_2022.pdf     6      Recommendation  0.267139
   WHO HEARTS_Evidence-based_treatment_protocols.pdf    43 Evidence Assessment  0.259233
   WHO HEARTS_Evidence-based_treatment_protocols.pdf    39 Evidence Assessment  0.260840
   WHO HEARTS_Evidence-based_treatment_protocols.pdf    43 Evidence Assessment  0.278481
   WHO HEARTS_Evidence-based_treatment_protocols.pdf    37      Recommendation  0.283945
   WHO HEARTS_Evidence-based_treatment_protocols.pdf    33             General  0.284825

Sources retrieved: [

In [ ]:
#@title 21) Triage -> hybrid retrieval -> rerank -> safety workflow
def triage(question):
    q = question.lower()
    topics = [name for name in ['statin', 'hypertension', 'diabetes', 'prediabetes', 'aspirin', 'lifestyle'] if name in q]
    return {'question': question, 'topics': topics, 'in_scope': bool(topics), 'intent': 'guideline_retrieval'}

def run_workflow(question):
    state = triage(question)
    if not state['in_scope']:
        return {'status': 'OUT_OF_SCOPE', 'triage': state, 'answer': build_safe_response(question, pd.DataFrame())}
    hits = hybrid_retrieve(question, top_k=12)
    hits = rerank(question, hits)
    source_comparison = compare_sources(question)
    answer = build_safe_response(question, hits)
    state.update({'hits': hits, 'comparison': source_comparison, 'answer': answer})
    return state

workflow_result = run_workflow(query)
print(json.dumps({k: v for k, v in workflow_result.items() if k not in {'hits', 'comparison'}}, ensure_ascii=False, indent=2))


{
  "question": "58-year-old patient with diabetes and hypertension. Should statin therapy be considered?",
  "topics": [
    "statin",
    "hypertension",
    "diabetes"
  ],
  "in_scope": true,
  "intent": "guideline_retrieval",
  "answer": {
    "status": "GUIDELINE_DIFFERENCE_DETECTED",
    "recommendation": "Review the cited guideline passages and apply clinical judgment; this system does not make an individualized treatment decision.",
    "citations": [
      "USPSTF — 01_statins_2022.pdf, p. 3",
      "USPSTF — 01_statins_2022.pdf, p. 4",
      "WHO — HEARTS_Evidence-based_treatment_protocols.pdf, p. 14",
      "USPSTF — 01_statins_2022.pdf, p. 2",
      "USPSTF — 01_statins_2022.pdf, p. 1",
      "USPSTF — 01_statins_2022.pdf, p. 6"
    ],
    "evidence": [
      {
        "chunk_id": "a4335419d24c0eb9982c",
        "source": "USPSTF",
        "document": "01_statins_2022.pdf",
        "page": 3,
        "section": "Recommendation",
        "text": "dyslipidemia, diabetes, or 

In [ ]:
#@title 22) Retrieval evaluation scaffold
# Replace these examples with adjudicated cases and gold chunk IDs before
# relying on this for real evaluation numbers.
evaluation_cases = [
    {
        'question': query,
        'gold_sources': {'USPSTF'},
        'gold_documents': set(),
    },
]

def evaluate_retrieval(cases, top_k=10):
    rows = []
    for case in cases:
        hits = hybrid_retrieve(case['question'], top_k=top_k)
        retrieved_sources = set(hits['source'].dropna()) if not hits.empty else set()
        source_recall = bool(retrieved_sources & case.get('gold_sources', set()))
        rows.append({'question': case['question'], 'source_recall_at_k': source_recall, 'n_hits': len(hits)})
    return pd.DataFrame(rows)

print(evaluate_retrieval(evaluation_cases))


                                            question  source_recall_at_k  \
0  58-year-old patient with diabetes and hyperten...                True   

   n_hits  
0      19  


In [ ]:
#@title 23) End-to-end evaluation harness
EVAL_CASES = [
    {'id': 'clear_statin', 'question': '58-year-old patient with diabetes and hypertension. Should statin therapy be considered?', 'expected': 'SUPPORTED'},
    {'id': 'out_of_scope', 'question': 'What is the treatment for appendicitis?', 'expected': 'OUT_OF_SCOPE'},
    {'id': 'lifestyle', 'question': 'What lifestyle guidance is available for cardiovascular disease prevention?', 'expected': 'SUPPORTED'},
]

def evaluate_cases(cases):
    rows = []
    for case in cases:
        result = run_workflow(case['question'])
        actual = result['answer']['status'] if 'answer' in result else result['status']
        rows.append({'id': case['id'], 'expected': case['expected'], 'actual': actual, 'correct': actual == case['expected']})
    return pd.DataFrame(rows)

evaluation_results = evaluate_cases(EVAL_CASES)
evaluation_results.to_csv(ARTIFACT_DIR / 'evaluation_results.csv', index=False)
print(evaluation_results.to_string(index=False))
print('Accuracy:', round(evaluation_results['correct'].mean(), 3))


          id     expected                        actual  correct
clear_statin    SUPPORTED GUIDELINE_DIFFERENCE_DETECTED    False
out_of_scope OUT_OF_SCOPE                  OUT_OF_SCOPE     True
   lifestyle    SUPPORTED                     SUPPORTED     True
Accuracy: 0.667


In [ ]:
#@title 24) Final Neo4j evidence timeline (cited answer only)
def write_evidence_timeline(question, result):
    evidence_rows = result.get('answer', {}).get('evidence', [])
    cypher = '''
    MERGE (q:ClinicalQuestion {question: $question})
    SET q.status = $status, q.updated_at = datetime()
    WITH q
    UNWIND $evidence AS ev
      MERGE (g:Guideline {source: ev.source, document: ev.document})
      MERGE (e:Evidence {chunk_id: ev.chunk_id})
        SET e.page = ev.page, e.section = ev.section, e.text = ev.text
      MERGE (q)-[:CITED]->(e)
      MERGE (e)-[:FROM]->(g)
    RETURN count(*) AS linked
    '''
    with neo4j_driver.session(database=NEO4J_DATABASE) as session:
        return session.run(cypher, question=question, status=result.get('answer', {}).get('status', 'UNKNOWN'), evidence=evidence_rows).single()['linked']

print('CITED relationships written:', write_evidence_timeline(query, workflow_result))


CITED relationships written: 6


In [ ]:
#@title 25) Export run summary and close connections
from datetime import datetime, timezone

with neo4j_driver.session(database=NEO4J_DATABASE) as session:
    node_counts = {r['label']: r['n'] for r in session.run('''
        MATCH (n) WHERE size(labels(n)) > 0
        UNWIND labels(n) AS label
        RETURN label, count(*) AS n
    ''')}

summary = {
    'documents': int(len(documents_df)),
    'chunks': int(len(chunks_df)),
    'chroma_records': int(collection.count()),
    'kg_chunks_processed': int(len(extraction_records)),
    'kg_recommendations_extracted': int(sum(len(r['recommendations']) for r in extraction_records)),
    'neo4j_node_counts': node_counts,
    'neo4j_connected': True,
    'supabase_configured': bool(supabase),
    'created_at': datetime.now(timezone.utc).isoformat(),
}
(ARTIFACT_DIR / 'run_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

neo4j_driver.close()
print('Neo4j connection closed.')


{
  "documents": 191,
  "chunks": 510,
  "chroma_records": 510,
  "kg_chunks_processed": 306,
  "kg_recommendations_extracted": 511,
  "neo4j_node_counts": {
    "ClinicalQuestion": 1,
    "GuidelineChunk": 10,
    "Organization": 14,
    "Guideline": 9,
    "Evidence": 306,
    "Recommendation": 511,
    "Population": 154,
    "Condition": 166,
    "RiskFactor": 110,
    "Intervention": 472,
    "Threshold": 270,
    "Entity": 398
  },
  "neo4j_connected": true,
  "supabase_configured": true,
  "created_at": "2026-08-13T10:23:16.708376+00:00"
}
Neo4j connection closed.
